# Stage 01 — MCSI Motorcycle Sales EDA
**Dashboard page:** MC Analysis · Tabs: Monthly Trend · By Model · By Color · By Year · RM/ASE/Geography · Dealer Performance

**Key business rule:** `sum(SlsVolQty)` per VIN = 1 → Sold, = 0 → Returned

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
clean = load("mcsi_clean.parquet")
vin   = load("mcsi_vin_status.parquet")
sold  = clean[clean["Status"]=="Sold"].copy()
ret   = clean[clean["Status"]=="Returned"].copy()
sold["date"]  = pd.to_datetime(sold.get("Billing Date", sold.get("Date")), errors="coerce")
sold["month"] = sold["date"].dt.to_period("M")
sold["year"]  = sold["date"].dt.year
REV = "Net Sales"

total_vins = len(vin); bikes_sold = int((vin["Status"]=="Sold").sum())
returns    = int((vin["Status"]=="Returned").sum())
return_rate = returns/total_vins*100
total_rev  = sold[REV].sum(); avg_per_unit = total_rev/bikes_sold
active_dealers = sold["Dealer"].nunique(); models_sold = sold["Model"].nunique()
avg_per_month  = bikes_sold / sold["month"].nunique()
date_min = sold["date"].min().strftime("%Y-%m-%d"); date_max = sold["date"].max().strftime("%Y-%m-%d")

print(f"Period           : {date_min} -> {date_max}")
print(f"Total VINs       : {total_vins:,}")
print(f"Bikes Sold       : {bikes_sold:,}  (avg {avg_per_month:,.0f}/month)")
print(f"Returns          : {returns}  ({return_rate:.2f}% return rate)")
print(f"Total Revenue    : {fmt_lkr(total_rev)}")
print(f"Avg LKR / unit   : {fmt_lkr(avg_per_unit)}")
print(f"Active Dealers   : {active_dealers}")
print(f"Models Sold      : {models_sold}")


## Tab 1 — Monthly Trend  (units sold bars + revenue LKR line)

In [ ]:
monthly = sold.groupby("month").agg(units=("SlsVolQty","sum"), revenue=(REV,"sum")).reset_index().sort_values("month")
monthly["ms"] = monthly["month"].astype(str)

fig, ax1 = plt.subplots(figsize=(13,5)); ax2 = ax1.twinx()
bars = ax1.bar(monthly["ms"], monthly["units"], color=PALETTE[0], alpha=0.75, label="Units Sold", zorder=2)
line,= ax2.plot(monthly["ms"], monthly["revenue"], color=PALETTE[1], lw=2.5, marker="o", ms=5, label="Revenue LKR", zorder=3)
ax1.set_title("Monthly Units Sold & Revenue\nBars=units (left) · Line=revenue LKR (right)", fontsize=12)
ax1.set_ylabel("Units Sold", color=PALETTE[0]); ax2.set_ylabel("Revenue (LKR)", color=PALETTE[1])
ax1.tick_params(axis="x", rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e9:.1f}B"))
ax2.spines["top"].set_visible(False)
ax1.legend([bars,line],["Units Sold","Revenue LKR"],loc="upper left",fontsize=9)
plt.tight_layout(); plt.show()
print(f"Return Rate: {return_rate:.2f}%  ({returns} returned / {total_vins:,} total VINs)")

## Tab 2 — By Model

In [ ]:
model_sum = sold.groupby("Model").agg(units=("SlsVolQty","sum"),revenue=(REV,"sum")).sort_values("units",ascending=False)
model_sum["avg_price"] = model_sum["revenue"]/model_sum["units"]
model_sum["rev_share"] = model_sum["revenue"]/model_sum["revenue"].sum()*100

fig, axes = plt.subplots(1,2,figsize=(15,5))
pivot = sold.groupby(["month","Model"])["SlsVolQty"].sum().unstack(fill_value=0)
pivot.index = pivot.index.astype(str)
pivot.plot(kind="bar",stacked=True,ax=axes[0],color=PALETTE[:len(pivot.columns)],edgecolor="white",lw=0.5)
axes[0].set_title("Monthly Unit Sales by Model"); axes[0].set_ylabel("Units")
axes[0].tick_params(axis="x",rotation=45); axes[0].legend(title="Model",fontsize=7)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))

model_sum["units"].sort_values().plot(kind="barh",ax=axes[1],color=PALETTE[:len(model_sum)],edgecolor="white")
axes[1].set_title("Total Units & Revenue Share by Model")
for i,(idx,row) in enumerate(model_sum.sort_values("units").iterrows()):
    axes[1].text(row["units"]+30,i,f"{fmt_lkr(row['revenue'])}  ({row['rev_share']:.1f}%)",va="center",fontsize=8)
plt.tight_layout(); plt.show()
print(model_sum.to_string())


## Tab 3 — By Color

In [ ]:
COLOR_KW = ["MAT DULL","MAT DARK","MATTE","METALLIC","BLACK","WHITE","BLUE","RED",
            "GRAY","GREY","ORANGE","GREEN","YELLOW","PURPLE","CYAN","SILVER"]
def extract_color(mat):
    mat = str(mat).upper()
    for kw in COLOR_KW:
        if kw in mat: return kw.split()[-1].title()
    return "Other"
sold["color"] = sold["Material"].apply(extract_color)
color_sum = sold.groupby("color").agg(units=("SlsVolQty","sum"),revenue=(REV,"sum")).sort_values("units",ascending=False)
color_sum["rev_share"] = color_sum["revenue"]/color_sum["revenue"].sum()*100

fig,axes = plt.subplots(1,2,figsize=(14,5))
color_sum["units"].plot(kind="bar",ax=axes[0],color=PALETTE[:len(color_sum)],edgecolor="white")
axes[0].set_title("Units Sold by Color"); axes[0].set_ylabel("Units")
for bar,val in zip(axes[0].patches,color_sum["units"]):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+30,f"{val:,}",ha="center",fontsize=8)
axes[1].pie(color_sum["units"],labels=color_sum.index,colors=PALETTE[:len(color_sum)],autopct="%1.1f%%",startangle=90)
axes[1].set_title("Color Share"); plt.tight_layout(); plt.show()

color_model = sold.groupby(["color","Model"])["SlsVolQty"].sum().unstack(fill_value=0)
fig,ax = plt.subplots(figsize=(12,4))
sns.heatmap(color_model,annot=True,fmt=".0f",cmap="Blues",linewidths=0.5,linecolor="white",ax=ax)
ax.set_title("Units — Color × Model Heatmap"); plt.tight_layout(); plt.show()


## Tab 4 — By Year

In [ ]:
year_sum = sold.groupby("year").agg(units=("SlsVolQty","sum"),revenue=(REV,"sum"),
    dealers=("Dealer","nunique"),models=("Model","nunique")).reset_index()
year_sum["avg_price"] = year_sum["revenue"]/year_sum["units"]
fig,axes = plt.subplots(1,3,figsize=(15,4))
for ax,col,lbl,fmt_fn in zip(axes,["units","revenue","avg_price"],
        ["Units","Revenue (LKR)","Avg LKR/unit"],
        [lambda x,_:f"{int(x):,}",lambda x,_:f"{x/1e9:.1f}B",lambda x,_:f"{x/1e3:.0f}K"]):
    year_sum.plot(x="year",y=col,kind="bar",ax=ax,color=PALETTE[0],edgecolor="white",legend=False)
    ax.set_title(lbl); ax.tick_params(axis="x",rotation=0)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_fn))
plt.suptitle("Year-over-Year Performance",fontsize=13,y=1.02); plt.tight_layout(); plt.show()
print(year_sum.to_string(index=False))


## Tab 5 — RM / ASE / Geography

In [ ]:
prov_sum = sold.groupby("Province").agg(units=("SlsVolQty","sum"),revenue=(REV,"sum"),
    dealers=("Dealer","nunique")).sort_values("units",ascending=False)
prov_sum["rev_share"] = prov_sum["revenue"]/prov_sum["revenue"].sum()*100
rm_sum = sold.groupby("RM").agg(units=("SlsVolQty","sum"),revenue=(REV,"sum")).sort_values("units",ascending=False)

fig,axes = plt.subplots(1,2,figsize=(15,5))
prov_sum["units"].plot(kind="bar",ax=axes[0],color=PALETTE[:len(prov_sum)],edgecolor="white")
axes[0].set_title("Units by Province (Province→RM→ASE→Dealer hierarchy)"); axes[0].set_ylabel("Units")
axes[0].tick_params(axis="x",rotation=45)
for bar,(idx,row) in zip(axes[0].patches,prov_sum.iterrows()):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+20,
        f"{row['units']:,}\n({row['rev_share']:.1f}%)",ha="center",va="bottom",fontsize=7)
rm_sum.head(15)["units"].sort_values().plot(kind="barh",ax=axes[1],color=PALETTE[0],edgecolor="white",alpha=0.8)
axes[1].set_title("Units by RM (Top 15)")
for i,(rm,row) in enumerate(rm_sum.head(15).sort_values("units").iterrows()):
    axes[1].text(row["units"]+5,i,fmt_lkr(row["revenue"]),va="center",fontsize=7)
plt.tight_layout(); plt.show()

prov_model = sold.groupby(["Province","Model"])["SlsVolQty"].sum().unstack(fill_value=0)
fig,ax = plt.subplots(figsize=(13,5))
sns.heatmap(prov_model,annot=True,fmt=".0f",cmap="YlOrRd",linewidths=0.5,linecolor="white",ax=ax)
ax.set_title("Province × Model Heatmap"); plt.tight_layout(); plt.show()
print(prov_sum.to_string())

## Tab 6 — Dealer Performance

In [ ]:
dlr = sold.groupby(["Dealer","Province"]).agg(units=("SlsVolQty","sum"),revenue=(REV,"sum"),
    models=("Model","nunique"),months_active=("month","nunique")).reset_index()
dlr["avg_monthly"] = dlr["units"]/dlr["months_active"]
dlr["avg_price"]   = dlr["revenue"]/dlr["units"]
dlr["rev_share"]   = dlr["revenue"]/dlr["revenue"].sum()*100
top20 = dlr.sort_values("units",ascending=False).head(20)

fig,axes = plt.subplots(1,2,figsize=(15,7))
top20.set_index("Dealer")["units"].sort_values().plot(kind="barh",ax=axes[0],color=PALETTE[0],edgecolor="white")
axes[0].set_title("Top 20 Dealers — Units Sold"); axes[0].set_xlabel("Units")
for i,(_,row) in enumerate(top20.sort_values("units").iterrows()):
    axes[0].text(row["units"]+5,i,f"{fmt_lkr(row['revenue'])} ({row['rev_share']:.1f}%)",va="center",fontsize=7)
top20.set_index("Dealer")["avg_monthly"].sort_values().plot(kind="barh",ax=axes[1],color=PALETTE[3],edgecolor="white")
axes[1].set_title("Top 20 Dealers — Avg Monthly Units"); axes[1].set_xlabel("Avg units/month")
plt.tight_layout(); plt.show()

# Pareto
d2 = dlr.sort_values("units",ascending=False).reset_index(drop=True)
d2["cum_pct"] = d2["units"].cumsum()/d2["units"].sum()*100
d2["dlr_pct"] = (d2.index+1)/len(d2)*100
fig,ax = plt.subplots(figsize=(11,4))
ax.plot(d2["dlr_pct"],d2["cum_pct"],color=PALETTE[0],lw=2.5)
ax.axhline(80,color=PALETTE[1],ls="--",alpha=0.7,label="80% of units")
top80 = d2[d2["cum_pct"]<=80]["dlr_pct"].max()
ax.axvline(top80,color=PALETTE[1],ls=":",alpha=0.5)
ax.set_title(f"Dealer Pareto: {top80:.1f}% of dealers = 80% of units")
ax.set_xlabel("Cumulative % of dealers"); ax.set_ylabel("Cumulative % of units"); ax.legend()
plt.tight_layout(); plt.show()
print(top20[["Dealer","Province","units","revenue","avg_monthly","avg_price","rev_share"]].to_string(index=False))


## Return Analysis

In [ ]:
print(f"Return rate : {return_rate:.4f}%  ({returns} of {total_vins:,} VINs)")
if len(ret)>0:
    ret2 = ret.copy(); ret2["date"]=pd.to_datetime(ret2.get("Billing Date",ret2.get("Date")),errors="coerce")
    ret2["month"]=ret2["date"].dt.to_period("M")
    combined=pd.concat([sold.groupby("month").size().rename("sold"),
                        ret2.groupby("month").size().rename("returns")],axis=1).fillna(0)
    combined["return_rate_pct"]=combined["returns"]/(combined["sold"]+combined["returns"])*100
    print(combined.to_string())
else:
    print("No returned VINs — 0.00% return rate.")
